In [2]:
import sys
import os

PROJECT_ROOT = "/content/drive/MyDrive/PFA"
SCRIPTS_ROOT = "/content/drive/MyDrive/PFA/ImageBind"
LIB_ROOT     = "/content/drive/MyDrive/PFA/ImageBind/ImageBind"

for p in [LIB_ROOT, SCRIPTS_ROOT, PROJECT_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
        print(f"✓ Linked: {p}")

✓ Linked: /content/drive/MyDrive/PFA/ImageBind/ImageBind
✓ Linked: /content/drive/MyDrive/PFA/ImageBind
✓ Linked: /content/drive/MyDrive/PFA


In [3]:
from shared_utils.video_processing import compute_segments, save_video_segment, get_video_info
from shared_utils.audio_processing import extract_audio_segment
from shared_utils.text_processing import extract_transcript_with_timestamps, align_text_to_segments
print("✓ All imports OK")

✓ All imports OK


In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/PFA/data/Obama_Yes_we_can.mp4"
assert os.path.exists(VIDEO_PATH), f"X Video not found: {VIDEO_PATH}"
print(f"✓ Video found: {VIDEO_PATH}")

✓ Video found: /content/drive/MyDrive/PFA/data/Obama_Yes_we_can.mp4


In [6]:
info = get_video_info(VIDEO_PATH)
print(f"Duration   : {info['duration']:.4f}s")
print(f"FPS        : {info['fps']:.4f}")
print(f"Resolution : {info['width']}x{info['height']}")

segments, fps, duration = compute_segments(VIDEO_PATH, window_size=2, stride=1)
print(f"\n✓ {len(segments)} segments computed")
print(f"  First : {segments[0]}")
print(f"  Last  : {segments[-1]}")

Duration   : 89.3333s
FPS        : 24.0000
Resolution : 1280x720

✓ 88 segments computed
  First : (0.0, 2.0)
  Last  : (87.0, 89.0)


In [ ]:
seg1, fps1, dur1 = compute_segments(VIDEO_PATH, window_size=2, stride=1)
seg2, fps2, dur2 = compute_segments(VIDEO_PATH, window_size=2, stride=1)

assert seg1 == seg2, "XX Segments differ between calls!"
assert fps1 == fps2, "XX FPS differs between calls!"
print(f"✓ compute_segments is deterministic ({len(seg1)} segments, {dur1:.4f}s)")

✓ compute_segments is deterministic (88 segments, 89.3333s)


In [ ]:
SR = 16000
expected = 2 * SR  # 2 seconds = 32000 samples

print("Checking first 5 audio segments:")
for i, (start, end) in enumerate(segments[:5]):
    audio = extract_audio_segment(VIDEO_PATH, start, end, sr=SR)
    diff = abs(len(audio) - expected)
    status = "✓" if diff <= 10 else "XX"
    print(f"  {status} Segment {i} [{start:.2f}s-{end:.2f}s]: {len(audio)} samples (Δ{diff})")

Checking first 5 audio segments:
  ✓ Segment 0 [0.00s-2.00s]: 32000 samples (Δ0)
  ✓ Segment 1 [1.00s-3.00s]: 32000 samples (Δ0)
  ✓ Segment 2 [2.00s-4.00s]: 32000 samples (Δ0)
  ✓ Segment 3 [3.00s-5.00s]: 32000 samples (Δ0)
  ✓ Segment 4 [4.00s-6.00s]: 32000 samples (Δ0)


In [12]:
from scripts.core import quick_load, create_engine
imagebind_mod, whisper_mod, device = quick_load(whisper_size="base")
engine = create_engine(imagebind_mod, device, whisper_mod)
print("✓ Models loaded")

LOADING MODELS

✓ GPU available: Tesla T4
VRAM: 15.6 GB
Loading ImageBind-huge model...
This downloads ~1.2GB on first run (2-3 minutes)...

✓ ImageBind model loaded successfully!
Device: cuda:0
GPU Memory Used: 4.88 GB
Loading Whisper model (size: base)...
✓ Whisper-base loaded successfully!

✓ All models loaded successfully!
✓ Models loaded


In [ ]:
transcript_segs, full_text = extract_transcript_with_timestamps(VIDEO_PATH, whisper_mod)
segment_tuples = [(None, s, e) for s, e in segments]
aligned_texts = align_text_to_segments(transcript_segs, segment_tuples)

print(f"Segments      : {len(segments)}")
print(f"Aligned texts : {len(aligned_texts)}")
assert len(aligned_texts) == len(segments), \
    f"X Mismatch: {len(aligned_texts)} texts vs {len(segments)} segments"
print(f"✓ Counts match")
print(f"  With text    : {sum(1 for t in aligned_texts if t)}")
print(f"  Without text : {sum(1 for t in aligned_texts if not t)}")

Detected language: English


100%|██████████| 8939/8939 [00:07<00:00, 1213.72frames/s]

Segments      : 88
Aligned texts : 88
✓ Counts match
  With text    : 86
  Without text : 2


In [ ]:
transcript_segs, full_text = extract_transcript_with_timestamps(VIDEO_PATH, whisper_mod)
segment_tuples = [(None, s, e) for s, e in segments]
aligned_texts = align_text_to_segments(transcript_segs, segment_tuples)

print(f"Segments      : {len(segments)}")
print(f"Aligned texts : {len(aligned_texts)}")
assert len(aligned_texts) == len(segments), \
    f"X Mismatch: {len(aligned_texts)} texts vs {len(segments)} segments"
print(f"✓ Counts match")
print(f"  With text    : {sum(1 for t in aligned_texts if t)}")
print(f"  Without text : {sum(1 for t in aligned_texts if not t)}")

Detected language: English


100%|██████████| 8939/8939 [00:02<00:00, 3538.19frames/s]

Segments      : 88
Aligned texts : 88
✓ Counts match
  With text    : 86
  Without text : 2


In [16]:
results, _ = engine.extract_video_features(VIDEO_PATH, window_size=2, stride=1)

v = sum(1 for r in results if r['vision_emb'] is not None)
a = sum(1 for r in results if r['audio_emb'] is not None)
t = sum(1 for r in results if r['text_emb'] is not None)

print(f"Total segments : {len(results)}")
print(f"Vision         : {v}")
print(f"Audio          : {a}")
print(f"Text           : {t}")
assert v == a == len(results), f" Modality mismatch V={v} A={a} total={len(results)}"
print("✓ All modalities aligned")

r0 = results[0]
print(f"\nShapes — Vision:{r0['vision_emb'].shape} Audio:{r0['audio_emb'].shape} Text:{r0['text_emb'].shape}")

Processing: /content/drive/MyDrive/PFA/data/Obama_Yes_we_can.mp4

Step 1: Extracting transcript with Whisper...
Detected language: English


100%|██████████| 8939/8939 [00:02<00:00, 3551.94frames/s]


✓ Extracted 20 text segments
✓ Total transcript length: 944 characters

Step 2: Segmenting video (window=2s, stride=1s)...
✓ Created 88 segments

Step 3: Aligning text to video segments...
✓ 86/88 segments have text

Step 4: Extracting trimodal embeddings...
Processing 88 segments (Vision + Audio + Text)...



Extracting features: 100%|██████████| 88/88 [05:20<00:00,  3.64s/it]


✓ Extraction complete!
  Processed: 88/88 segments
  Each segment has:
    - Vision embedding: (1024,)
    - Audio embedding: (1024,)
    - Text embedding: (1024,)
Total segments : 88
Vision         : 88
Audio          : 88
Text           : 88
✓ All modalities aligned

Shapes — Vision:(1, 1024) Audio:(1, 1024) Text:(1, 1024)
